## RCNN

In [ ]:
class RCNN(nn.Module):
    def __init__(self, num_classes):
        super(RCNN, self).__init__()
        # Load pre-trained VGG16 as the feature extractor
        vgg = vgg16(weights=VGG16_Weights.DEFAULT)
        self.feature_extractor = nn.Sequential(*list(vgg.features.children()))
        self.avgpool = nn.AdaptiveAvgPool2d((7, 7))

        # Classifier on top of the features
        self.classifier = nn.Sequential(
            nn.Linear(512 * 7 * 7, 4096),
            nn.ReLU(True),
            nn.Dropout(),
            nn.Linear(4096, 4096),
            nn.ReLU(True),
            nn.Dropout(),
            nn.Linear(4096, num_classes)
        )

        # Bounding box regressor
        self.bbox_regressor = nn.Sequential(
            nn.Linear(512 * 7 * 7, 4096),
            nn.ReLU(True),
            nn.Dropout(),
            nn.Linear(4096, 4 * num_classes)  # (x, y, w, h) for each class
        )

    def forward(self, x):
        # Extract features
        features = self.feature_extractor(x)
        features = self.avgpool(features)
        # Flatten the features
        features = features.view(features.size(0), -1)
        # Get class scores and bounding box predictions
        class_scores = self.classifier(features)
        bbox_preds = self.bbox_regressor(features)

        return class_scores, bbox_preds

# Selective Search for Region Proposals
class SelectiveSearch:
    def __init__(self):
        # Initialize selective search segmentation
        self.ss = cv2.ximgproc.segmentation.createSelectiveSearchSegmentation()

    def process_image(self, image, max_proposals=2000):
        """Generate region proposals using Selective Search"""
        # Convert PIL Image to OpenCV format if necessary
        if isinstance(image, Image.Image):
            image = np.array(image)
            image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        self.ss.setBaseImage(image)
        # Set to fast mode for quicker processing
        self.ss.switchToSelectiveSearchFast()
        # Get region proposals
        rects = self.ss.process()

        # Limit the number of proposals
        if len(rects) > max_proposals:
            rects = rects[:max_proposals]

        return rects

## Fast RCNN

In [ ]:
class FastRCNN(nn.Module):
    def __init__(self, num_classes):
        super(FastRCNN, self).__init__()
        # Load pre-trained VGG16 as the feature extractor
        vgg = vgg16(weights=VGG16_Weights.DEFAULT)
        self.feature_extractor = nn.Sequential(*list(vgg.features.children()))

        # ROI Pooling layer
        self.roi_pool = RoIPool(output_size=(7, 7), spatial_scale=1/16)

        # Classifier head
        self.classifier = nn.Sequential(
            nn.Linear(512 * 7 * 7, 4096),
            nn.ReLU(True),
            nn.Dropout(),
            nn.Linear(4096, 4096),
            nn.ReLU(True),
            nn.Dropout(),
        )

        # Final prediction layers
        self.cls_score = nn.Linear(4096, num_classes)
        self.bbox_pred = nn.Linear(4096, 4 * num_classes)  # (x, y, w, h) for each class

    def forward(self, image, rois):
        # Extract features from the entire image once
        features = self.feature_extractor(image)  # [batch_size, 512, h/16, w/16]

        # Apply ROI pooling for each region proposal
        # rois shape: [num_rois, 5] where each row is (batch_idx, x1, y1, x2, y2)
        pooled_features = self.roi_pool(features, rois)

        # Flatten the features
        pooled_features = pooled_features.view(pooled_features.size(0), -1)

        # Feed into the classifier head
        fc_features = self.classifier(pooled_features)

        # Get class scores and bounding box predictions
        class_scores = self.cls_score(fc_features)
        bbox_preds = self.bbox_pred(fc_features)

        return class_scores, bbox_preds

# ROI Pooling implementation
class RoIPool(nn.Module):
    def __init__(self, output_size, spatial_scale):
        super(RoIPool, self).__init__()
        self.output_size = output_size
        self.spatial_scale = spatial_scale

    def forward(self, features, rois):
        """
        Args:
            features: [batch_size, channels, height, width]
            rois: [num_rois, 5] where each row is (batch_idx, x1, y1, x2, y2)
        """
        return roi_pool(features, rois, self.output_size, self.spatial_scale)


# Selective Search for Region Proposals (same as in R-CNN)

## FasterRCNN

In [ ]:
class FasterRCNN(nn.Module):
    def __init__(self, num_classes):
        super(FasterRCNN, self).__init__()
        # Load pre-trained VGG16 as the backbone
        vgg = vgg16(weights=VGG16_Weights.DEFAULT)
        features = list(vgg.features.children())

        # Use conv1 to conv5 from VGG16 as backbone
        self.backbone = nn.Sequential(*features[:30])  # up to conv5

        # Region Proposal Network
        self.rpn = RegionProposalNetwork(in_channels=512, mid_channels=512)

        # ROI Pooling layer
        self.roi_pool = RoIPool(output_size=(7, 7), spatial_scale=1/16)

        # Classifier head (same as Fast R-CNN)
        self.classifier = nn.Sequential(
            nn.Linear(512 * 7 * 7, 4096),
            nn.ReLU(True),
            nn.Dropout(),
            nn.Linear(4096, 4096),
            nn.ReLU(True),
            nn.Dropout(),
        )

        # Final prediction layers
        self.cls_score = nn.Linear(4096, num_classes)
        self.bbox_pred = nn.Linear(4096, 4 * num_classes)  # (x, y, w, h) for each class

    def forward(self, images, image_shapes):
        """
        Args:
            images: Tensor of shape [batch_size, channels, height, width]
            image_shapes: List of (height, width) for each image in the batch
        """
        # Extract features from the backbone network
        features = self.backbone(images)

        # Generate region proposals
        rpn_objectness, rpn_bbox_reg, rois = self.rpn(features, image_shapes)

        # Apply ROI pooling
        pooled_features = self.roi_pool(features, rois)

        # Flatten the features
        pooled_features = pooled_features.view(pooled_features.size(0), -1)

        # Feed into the classifier head
        fc_features = self.classifier(pooled_features)

        # Get class scores and bounding box predictions
        class_scores = self.cls_score(fc_features)
        bbox_preds = self.bbox_pred(fc_features)

        return class_scores, bbox_preds, rpn_objectness, rpn_bbox_reg, rois

# Region Proposal Network
class RegionProposalNetwork(nn.Module):
    def __init__(self, in_channels=512, mid_channels=512, anchor_ratios=[0.5, 1, 2],
                 anchor_scales=[8, 16, 32], feat_stride=16):
        super(RegionProposalNetwork, self).__init__()

        self.feat_stride = feat_stride

        # Generate anchor boxes
        self.anchor_generator = AnchorGenerator(
            anchor_ratios=anchor_ratios,
            anchor_scales=anchor_scales
        )

        # Number of anchor boxes per location
        num_anchors = len(anchor_ratios) * len(anchor_scales)

        # 3x3 conv layer
        self.conv = nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1)

        # 1x1 conv for objectness classification (foreground vs background)
        self.cls_score = nn.Conv2d(mid_channels, num_anchors * 2, kernel_size=1)

        # 1x1 conv for bounding box regression
        self.bbox_pred = nn.Conv2d(mid_channels, num_anchors * 4, kernel_size=1)

        # Proposal layer
        self.proposal_layer = ProposalLayer(
            feat_stride=feat_stride,
            anchor_generator=self.anchor_generator
        )

    def forward(self, features, image_shapes):
        """
        Args:
            features: [batch_size, channels, height, width]
            image_shapes: List of (height, width) for each image in the batch
        """
        # Shared convolutional layer
        shared_features = F.relu(self.conv(features))

        # Objectness classification
        rpn_cls_score = self.cls_score(shared_features)

        # Reshape from [batch_size, 2*num_anchors, h, w] to [batch_size, 2, num_anchors*h*w]
        batch_size = rpn_cls_score.size(0)
        channels = rpn_cls_score.size(1)
        h, w = rpn_cls_score.size(2), rpn_cls_score.size(3)
        rpn_cls_score = rpn_cls_score.permute(0, 2, 3, 1).contiguous().view(batch_size, -1, 2)

        # Apply softmax to get objectness probability
        rpn_objectness = F.softmax(rpn_cls_score, dim=2)

        # Bounding box regression
        rpn_bbox_reg = self.bbox_pred(shared_features)

        # Reshape from [batch_size, 4*num_anchors, h, w] to [batch_size, 4, num_anchors*h*w]
        rpn_bbox_reg = rpn_bbox_reg.permute(0, 2, 3, 1).contiguous().view(batch_size, -1, 4)

        # Generate proposals
        rois = self.proposal_layer(rpn_objectness, rpn_bbox_reg, features, image_shapes)

        return rpn_objectness, rpn_bbox_reg, rois

# Anchor Generator
class AnchorGenerator:
    def __init__(self, anchor_ratios=[0.5, 1, 2], anchor_scales=[8, 16, 32]):
        self.anchor_ratios = anchor_ratios
        self.anchor_scales = anchor_scales

    def generate_anchors(self, feature_map_size, feature_stride):
        """
        Generate anchor boxes for each position in the feature map
        Args:
            feature_map_size: (height, width) of the feature map
            feature_stride: stride between feature map and original image
        """
        # All possible anchor boxes for a single position
        base_anchors = self._generate_base_anchors()

        # Generate grid of anchor centers
        height, width = feature_map_size
        shift_x = np.arange(0, width) * feature_stride
        shift_y = np.arange(0, height) * feature_stride
        shift_x, shift_y = np.meshgrid(shift_x, shift_y)
        shifts = np.vstack((shift_x.ravel(), shift_y.ravel(),
                            shift_x.ravel(), shift_y.ravel())).transpose()

        # Add each anchor to each position
        A = len(base_anchors)
        K = len(shifts)
        all_anchors = base_anchors.reshape((1, A, 4)) + shifts.reshape((K, 1, 4))
        all_anchors = all_anchors.reshape((K * A, 4))

        return all_anchors

    def _generate_base_anchors(self):
        """Generate base anchor boxes of different sizes and aspect ratios"""
        base_anchors = []
        for scale in self.anchor_scales:
            for ratio in self.anchor_ratios:
                h = scale * np.sqrt(ratio)
                w = scale / np.sqrt(ratio)

                # (x_center, y_center, width, height) format
                base_anchors.append([-w/2, -h/2, w/2, h/2])

        return np.array(base_anchors)

# Proposal Layer
class ProposalLayer:
    def __init__(self, feat_stride, anchor_generator,
                 pre_nms_top_n=6000, post_nms_top_n=300,
                 nms_thresh=0.7, min_size=16):
        self.feat_stride = feat_stride
        self.anchor_generator = anchor_generator
        self.pre_nms_top_n = pre_nms_top_n
        self.post_nms_top_n = post_nms_top_n
        self.nms_thresh = nms_thresh
        self.min_size = min_size

    def forward(self, rpn_objectness, rpn_bbox_reg, features, image_shapes):
        """
        Generate ROIs from RPN outputs
        Args:
            rpn_objectness: [batch_size, num_anchors*h*w, 2]
            rpn_bbox_reg: [batch_size, num_anchors*h*w, 4]
            features: [batch_size, channels, h, w]
            image_shapes: List of (height, width) for each image in the batch
        """
        batch_size = features.size(0)
        feat_height, feat_width = features.size(2), features.size(3)

        # Generate anchors
        anchors = self.anchor_generator.generate_anchors(
            (feat_height, feat_width), self.feat_stride)

        # Convert anchors to torch tensor
        anchors = torch.from_numpy(anchors).to(features.device).float()

        # Process each image in the batch
        all_rois = []
        for i in range(batch_size):
            # Get scores (objectness probability)
            scores = rpn_objectness[i, :, 1]  # foreground probability

            # Get bbox regression deltas
            bbox_deltas = rpn_bbox_reg[i]

            # Apply deltas to anchors to get proposals
            proposals = self._apply_deltas(anchors, bbox_deltas)

            # Clip proposals to image boundaries
            img_h, img_w = image_shapes[i]
            proposals = self._clip_boxes(proposals, img_h, img_w)

            # Remove small proposals
            keep = self._filter_small_boxes(proposals, self.min_size)
            proposals = proposals[keep]
            scores = scores[keep]

            # Non-maximum suppression
            keep = self._nms(proposals, scores, self.nms_thresh)
            keep = keep[:self.post_nms_top_n]
            proposals = proposals[keep]

            # Add batch index
            batch_idx = torch.full((proposals.size(0), 1), i, dtype=torch.float32, device=proposals.device)
            rois = torch.cat((batch_idx, proposals), dim=1)
            all_rois.append(rois)

        # Combine all rois
        rois = torch.cat(all_rois, dim=0)
        return rois

    def _apply_deltas(self, boxes, deltas):
        """Apply bbox regression deltas to anchor boxes"""
        # Convert boxes from (x1, y1, x2, y2) to (x_center, y_center, width, height)
        widths = boxes[:, 2] - boxes[:, 0]
        heights = boxes[:, 3] - boxes[:, 1]
        x_centers = boxes[:, 0] + 0.5 * widths
        y_centers = boxes[:, 1] + 0.5 * heights

        # Apply deltas
        dx = deltas[:, 0]
        dy = deltas[:, 1]
        dw = deltas[:, 2]
        dh = deltas[:, 3]

        # Scale by box dimensions
        dx = dx * widths
        dy = dy * heights
        dw = dw * widths
        dh = dh * heights

        # Update box parameters
        pred_x_centers = x_centers + dx
        pred_y_centers = y_centers + dy
        pred_widths = widths * torch.exp(dw)
        pred_heights = heights * torch.exp(dh)

        # Convert back to (x1, y1, x2, y2)
        pred_boxes = torch.zeros_like(boxes)
        pred_boxes[:, 0] = pred_x_centers - 0.5 * pred_widths
        pred_boxes[:, 1] = pred_y_centers - 0.5 * pred_heights
        pred_boxes[:, 2] = pred_x_centers + 0.5 * pred_widths
        pred_boxes[:, 3] = pred_y_centers + 0.5 * pred_heights

        return pred_boxes

    def _clip_boxes(self, boxes, img_h, img_w):
        """Clip boxes to image boundaries"""
        boxes[:, 0] = torch.clamp(boxes[:, 0], min=0, max=img_w)
        boxes[:, 1] = torch.clamp(boxes[:, 1], min=0, max=img_h)
        boxes[:, 2] = torch.clamp(boxes[:, 2], min=0, max=img_w)
        boxes[:, 3] = torch.clamp(boxes[:, 3], min=0, max=img_h)
        return boxes

    def _filter_small_boxes(self, boxes, min_size):
        """Remove boxes with width or height smaller than min_size"""
        widths = boxes[:, 2] - boxes[:, 0]
        heights = boxes[:, 3] - boxes[:, 1]
        keep = (widths >= min_size) & (heights >= min_size)
        return keep

    def _nms(self, boxes, scores, threshold):
        """Non-maximum suppression"""
        x1 = boxes[:, 0]
        y1 = boxes[:, 1]
        x2 = boxes[:, 2]
        y2 = boxes[:, 3]

        areas = (x2 - x1) * (y2 - y1)
        order = scores.argsort(descending=True)

        keep = []
        while order.numel() > 0:
            if order.numel() == 1:
                keep.append(order.item())
                break

            i = order[0].item()
            keep.append(i)

            # Compute IoU of the current box with the rest
            xx1 = torch.max(x1[i], x1[order[1:]])
            yy1 = torch.max(y1[i], y1[order[1:]])
            xx2 = torch.min(x2[i], x2[order[1:]])
            yy2 = torch.min(y2[i], y2[order[1:]])

            w = torch.clamp(xx2 - xx1, min=0)
            h = torch.clamp(yy2 - yy1, min=0)
            inter = w * h

            iou = inter / (areas[i] + areas[order[1:]] - inter)

            # Keep boxes with IoU below threshold
            inds = torch.where(iou <= threshold)[0]
            order = order[inds + 1]

        return torch.tensor(keep, dtype=torch.long, device=boxes.device)

# ROI Pooling implementation
class RoIPool(nn.Module):
    def __init__(self, output_size, spatial_scale):
        super(RoIPool, self).__init__()
        self.output_size = output_size
        self.spatial_scale = spatial_scale

    def forward(self, features, rois):
        """
        Args:
            features: [batch_size, channels, height, width]
            rois: [num_rois, 5] where each row is (batch_idx, x1, y1, x2, y2)
        """
        return roi_pool(features, rois, self.output_size, self.spatial_scale)

## YOLO v1

In [ ]:
# YOLO v1 Implementation
class YOLOv1(nn.Module):
    def __init__(self, num_classes, S=7, B=2):
        """
        YOLO v1 implementation

        Args:
            num_classes (int): Number of classes to detect
            S (int): Grid size (S x S)
            B (int): Number of bounding boxes per grid cell
        """
        super(YOLOv1, self).__init__()
        self.S = S  # Grid size
        self.B = B  # Number of bounding boxes per grid cell
        self.num_classes = num_classes

        # Darknet-inspired feature extractor (similar to paper architecture)
        self.features = nn.Sequential(
            # Initial convolution layer
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3),
            nn.LeakyReLU(0.1),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Conv layers
            nn.Conv2d(64, 192, kernel_size=3, padding=1),
            nn.LeakyReLU(0.1),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(192, 128, kernel_size=1),
            nn.LeakyReLU(0.1),
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.LeakyReLU(0.1),
            nn.Conv2d(256, 256, kernel_size=1),
            nn.LeakyReLU(0.1),
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.LeakyReLU(0.1),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(512, 256, kernel_size=1),
            nn.LeakyReLU(0.1),
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.LeakyReLU(0.1),
            nn.Conv2d(512, 256, kernel_size=1),
            nn.LeakyReLU(0.1),
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.LeakyReLU(0.1),
            nn.Conv2d(512, 512, kernel_size=1),
            nn.LeakyReLU(0.1),
            nn.Conv2d(512, 1024, kernel_size=3, padding=1),
            nn.LeakyReLU(0.1),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(1024, 512, kernel_size=1),
            nn.LeakyReLU(0.1),
            nn.Conv2d(512, 1024, kernel_size=3, padding=1),
            nn.LeakyReLU(0.1),
            nn.Conv2d(1024, 512, kernel_size=1),
            nn.LeakyReLU(0.1),
            nn.Conv2d(512, 1024, kernel_size=3, padding=1),
            nn.LeakyReLU(0.1),
            nn.Conv2d(1024, 1024, kernel_size=3, padding=1),
            nn.LeakyReLU(0.1),
            nn.Conv2d(1024, 1024, kernel_size=3, stride=2, padding=1),
            nn.LeakyReLU(0.1),

            nn.Conv2d(1024, 1024, kernel_size=3, padding=1),
            nn.LeakyReLU(0.1),
            nn.Conv2d(1024, 1024, kernel_size=3, padding=1),
            nn.LeakyReLU(0.1),
        )

        # Final output layers
        # Output shape: S x S x (B*5 + C)
        # Where 5 = (x, y, w, h, confidence) for each bounding box
        output_size = self.S * self.S * (self.B * 5 + self.num_classes)

        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1024 * S * S, 4096),
            nn.LeakyReLU(0.1),
            nn.Dropout(0.5),
            nn.Linear(4096, output_size),
            nn.Sigmoid()  # Normalize all outputs between 0 and 1
        )

    def forward(self, x):
        """
        Forward pass

        Args:
            x: Input tensor of shape (batch_size, 3, 448, 448)

        Returns:
            Tensor of shape (batch_size, S, S, B*5+num_classes)
        """
        # Feature extraction
        x = self.features(x)

        # Fully connected layers
        x = self.fc_layers(x)

        # Reshape to (batch_size, S, S, B*5+num_classes)
        batch_size = x.size(0)
        x = x.view(batch_size, self.S, self.S, self.B * 5 + self.num_classes)

        return x

    def predict(self, x, conf_threshold=0.5, nms_threshold=0.4):
        """
        Make predictions with non-maximum suppression

        Args:
            x: Input image tensor
            conf_threshold: Confidence threshold for detections
            nms_threshold: IoU threshold for non-maximum suppression

        Returns:
            List of detections with [class_idx, confidence, x, y, w, h]
        """
        # Forward pass
        output = self.forward(x)
        batch_size = output.size(0)
        predictions = []

        # Process each image in the batch
        for b in range(batch_size):
            img_preds = []

            # Process grid cells
            for i in range(self.S):
                for j in range(self.S):
                    # Get cell info
                    cell_info = output[b, i, j]

                    # Process each bounding box
                    for box_idx in range(self.B):
                        # Get box confidence
                        box_start = box_idx * 5
                        confidence = cell_info[box_start + 4]

                        # Only consider boxes above threshold
                        if confidence > conf_threshold:
                            # Get box coordinates
                            x_center = (cell_info[box_start] + j) / self.S
                            y_center = (cell_info[box_start + 1] + i) / self.S
                            width = cell_info[box_start + 2]
                            height = cell_info[box_start + 3]

                            # Get class probabilities
                            class_offset = self.B * 5
                            class_probs = cell_info[class_offset:class_offset + self.num_classes]
                            class_idx = torch.argmax(class_probs)
                            class_score = class_probs[class_idx]

                            # Calculate final score
                            final_score = confidence * class_score

                            if final_score > conf_threshold:
                                # Convert to [class, confidence, x, y, w, h] format
                                img_preds.append([class_idx.item(), final_score.item(),
                                                 x_center.item(), y_center.item(),
                                                 width.item(), height.item()])

            # Apply non-maximum suppression
            img_preds = self._non_max_suppression(img_preds, nms_threshold)
            predictions.append(img_preds)

        return predictions

    def _non_max_suppression(self, boxes, iou_threshold):
        """
        Perform non-maximum suppression

        Args:
            boxes: List of [class_idx, confidence, x, y, w, h]
            iou_threshold: IoU threshold for NMS

        Returns:
            List of filtered boxes
        """
        if not boxes:
            return []

        # Sort by confidence
        boxes.sort(key=lambda x: x[1], reverse=True)

        # Apply NMS
        new_boxes = []
        while boxes:
            chosen_box = boxes.pop(0)
            new_boxes.append(chosen_box)

            boxes = [box for box in boxes if (box[0] != chosen_box[0] or
                     self._calculate_iou(chosen_box[2:], box[2:]) < iou_threshold)]

        return new_boxes

    def _calculate_iou(self, box1, box2):
        """
        Calculate IoU between two boxes

        Args:
            box1: [x_center, y_center, width, height]
            box2: [x_center, y_center, width, height]

        Returns:
            IoU value
        """
        # Convert to [x1, y1, x2, y2] format
        box1_x1 = box1[0] - box1[2] / 2
        box1_y1 = box1[1] - box1[3] / 2
        box1_x2 = box1[0] + box1[2] / 2
        box1_y2 = box1[1] + box1[3] / 2

        box2_x1 = box2[0] - box2[2] / 2
        box2_y1 = box2[1] - box2[3] / 2
        box2_x2 = box2[0] + box2[2] / 2
        box2_y2 = box2[1] + box2[3] / 2

        # Calculate intersection area
        x1 = max(box1_x1, box2_x1)
        y1 = max(box1_y1, box2_y1)
        x2 = min(box1_x2, box2_x2)
        y2 = min(box1_y2, box2_y2)

        # Check if boxes overlap
        if x2 < x1 or y2 < y1:
            return 0.0

        intersection_area = (x2 - x1) * (y2 - y1)

        # Calculate union area
        box1_area = (box1_x2 - box1_x1) * (box1_y2 - box1_y1)
        box2_area = (box2_x2 - box2_x1) * (box2_y2 - box2_y1)
        union_area = box1_area + box2_area - intersection_area

        return intersection_area / union_area

## YOLO v8

In [ ]:
# YOLOv8 Implementation
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1):
        super().__init__()
        padding = kernel_size // 2
        self.conv = nn.Conv2d(
            in_channels, out_channels, kernel_size, stride, padding, bias=False
        )
        self.bn = nn.BatchNorm2d(out_channels)
        self.silu = nn.SiLU()

    def forward(self, x):
        return self.silu(self.bn(self.conv(x)))


class ResidualBlock(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        mid_channels = in_channels // 2
        self.conv1 = ConvBlock(in_channels, mid_channels, kernel_size=1)
        self.conv2 = ConvBlock(mid_channels, in_channels, kernel_size=3)

    def forward(self, x):
        return x + self.conv2(self.conv1(x))


class CSPLayer(nn.Module):
    def __init__(self, in_channels, out_channels, num_blocks=1):
        super().__init__()
        mid_channels = out_channels // 2

        self.conv1 = ConvBlock(in_channels, mid_channels, kernel_size=1)
        self.conv2 = ConvBlock(in_channels, mid_channels, kernel_size=1)

        self.blocks = nn.Sequential(*[
            ResidualBlock(mid_channels) for _ in range(num_blocks)
        ])

        self.conv3 = ConvBlock(mid_channels * 2, out_channels, kernel_size=1)

    def forward(self, x):
        x1 = self.conv1(x)
        x2 = self.blocks(self.conv2(x))
        x3 = torch.cat([x2, x1], dim=1)
        return self.conv3(x3)


class SPPFLayer(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        mid_channels = in_channels // 2
        self.conv1 = ConvBlock(in_channels, mid_channels, kernel_size=1)
        self.conv2 = ConvBlock(mid_channels * 4, out_channels, kernel_size=1)
        self.maxpool = nn.MaxPool2d(kernel_size=5, stride=1, padding=2)

    def forward(self, x):
        x = self.conv1(x)
        y1 = self.maxpool(x)
        y2 = self.maxpool(y1)
        y3 = self.maxpool(y2)
        concat = torch.cat([x, y1, y2, y3], dim=1)
        return self.conv2(concat)


class YOLOv8Backbone(nn.Module):
    def __init__(self):
        super().__init__()
        # Initial Conv
        self.conv1 = ConvBlock(3, 64, kernel_size=3, stride=2)
        self.conv2 = ConvBlock(64, 128, kernel_size=3, stride=2)

        # CSP Stage 1
        self.csp1 = CSPLayer(128, 128, num_blocks=1)
        self.conv3 = ConvBlock(128, 256, kernel_size=3, stride=2)

        # CSP Stage 2
        self.csp2 = CSPLayer(256, 256, num_blocks=2)
        self.conv4 = ConvBlock(256, 512, kernel_size=3, stride=2)

        # CSP Stage 3
        self.csp3 = CSPLayer(512, 512, num_blocks=3)
        self.conv5 = ConvBlock(512, 1024, kernel_size=3, stride=2)

        # CSP Stage 4
        self.csp4 = CSPLayer(1024, 1024, num_blocks=1)
        self.sppf = SPPFLayer(1024, 1024)

    def forward(self, x):
        # Feature extraction
        x = self.conv1(x)
        x = self.conv2(x)

        # Stage 1
        x = self.csp1(x)
        x = self.conv3(x)

        # Stage 2
        x1 = self.csp2(x)    # P3
        x = self.conv4(x1)

        # Stage 3
        x2 = self.csp3(x)    # P4
        x = self.conv5(x2)

        # Stage 4
        x = self.csp4(x)
        x3 = self.sppf(x)    # P5

        return x1, x2, x3


class YOLOv8Neck(nn.Module):
    def __init__(self):
        super().__init__()
        # Upsampling path
        self.conv1 = ConvBlock(1024, 512, kernel_size=1)
        self.upsample = nn.Upsample(scale_factor=2, mode='nearest')
        self.csp1 = CSPLayer(1024, 512, num_blocks=1)

        self.conv2 = ConvBlock(512, 256, kernel_size=1)
        self.csp2 = CSPLayer(512, 256, num_blocks=1)

        # Downsampling path
        self.conv3 = ConvBlock(256, 256, kernel_size=3, stride=2)
        self.csp3 = CSPLayer(512, 512, num_blocks=1)

        self.conv4 = ConvBlock(512, 512, kernel_size=3, stride=2)
        self.csp4 = CSPLayer(1024, 1024, num_blocks=1)

    def forward(self, features):
        x1, x2, x3 = features

        # FPN top-down pathway
        p5 = self.conv1(x3)
        p5_upsample = self.upsample(p5)
        p4 = torch.cat([p5_upsample, x2], dim=1)
        p4 = self.csp1(p4)

        p4_conv = self.conv2(p4)
        p4_upsample = self.upsample(p4_conv)
        p3 = torch.cat([p4_upsample, x1], dim=1)
        p3 = self.csp2(p3)

        # PAN bottom-up pathway
        p3_downsample = self.conv3(p3)
        p4 = torch.cat([p3_downsample, p4_conv], dim=1)
        p4 = self.csp3(p4)

        p4_downsample = self.conv4(p4)
        p5 = torch.cat([p4_downsample, p5], dim=1)
        p5 = self.csp4(p5)

        return p3, p4, p5


class YOLOv8Head(nn.Module):
    def __init__(self, num_classes, reg_max=16):
        super().__init__()
        self.num_classes = num_classes
        self.reg_max = reg_max

        # Detection head for P3 (feature level 0)
        self.head_p3_cls = nn.Conv2d(256, num_classes, kernel_size=1)
        self.head_p3_reg = nn.Conv2d(256, 4 * (reg_max + 1), kernel_size=1)

        # Detection head for P4 (feature level 1)
        self.head_p4_cls = nn.Conv2d(512, num_classes, kernel_size=1)
        self.head_p4_reg = nn.Conv2d(512, 4 * (reg_max + 1), kernel_size=1)

        # Detection head for P5 (feature level 2)
        self.head_p5_cls = nn.Conv2d(1024, num_classes, kernel_size=1)
        self.head_p5_reg = nn.Conv2d(1024, 4 * (reg_max + 1), kernel_size=1)

        # Project for dbox distribution (for better regression)
        self.project = nn.Parameter(torch.linspace(0, self.reg_max, self.reg_max + 1), requires_grad=False)

    def forward(self, features):
        p3, p4, p5 = features

        # Classification and distribution-based box regression for each feature map
        cls_p3, dis_p3 = self.head_p3_cls(p3), self.head_p3_reg(p3)
        cls_p4, dis_p4 = self.head_p4_cls(p4), self.head_p4_reg(p4)
        cls_p5, dis_p5 = self.head_p5_cls(p5), self.head_p5_reg(p5)

        # Return output by feature level
        return [
            (cls_p3, dis_p3),
            (cls_p4, dis_p4),
            (cls_p5, dis_p5)
        ]

    def decode_outputs(self, outputs, dtype=torch.float32):
        grids = []
        strides = []

        # Define strides for different feature maps
        all_strides = [8, 16, 32]  # Strides for P3, P4, P5

        all_cls_preds = []
        all_bbox_preds = []

        for i, (cls_pred, reg_pred) in enumerate(outputs):
            batch_size, _, h, w = cls_pred.shape

            # Create grid coordinates
            yv, xv = torch.meshgrid([torch.arange(h), torch.arange(w)], indexing='ij')
            grid = torch.stack((xv, yv), 2).view(1, h, w, 2).to(cls_pred.device)
            stride = torch.full((1, h, w, 1), all_strides[i], dtype=dtype, device=cls_pred.device)

            # Save grid and stride for later use
            grids.append(grid)
            strides.append(stride)

            # Process classification predictions
            cls_pred = cls_pred.sigmoid()

            # Process regression predictions (distribution-based)
            reg_pred = reg_pred.reshape(batch_size, 4, self.reg_max + 1, h, w)
            reg_pred = reg_pred.permute(0, 1, 3, 4, 2).reshape(batch_size, 4, -1)
            reg_pred = F.softmax(reg_pred, dim=-1)
            reg_pred = reg_pred @ self.project.to(reg_pred.device)
            reg_pred = reg_pred.reshape(batch_size, 4, h, w)

            # Adjust box predictions to be relative to cell position
            pred_xy = (cls_pred.sigmoid().unsqueeze(1) * reg_pred.unsqueeze(0)).view(batch_size, -1, h, w)

            all_cls_preds.append(cls_pred.flatten(2).transpose(1, 2))
            all_bbox_preds.append(reg_pred.flatten(2).transpose(1, 2))

        # Concatenate predictions from all feature maps
        cls_preds = torch.cat(all_cls_preds, dim=1)
        box_preds = torch.cat(all_bbox_preds, dim=1)

        # Adjust box predictions with grid coordinates and strides
        grids = torch.cat([grid.view(-1, 2) for grid in grids], dim=0)
        strides = torch.cat([stride.view(-1, 1) for stride in strides], dim=0)

        # Convert box predictions to absolute coordinates
        box_preds[..., :2] = grids + box_preds[..., :2]  # xy
        box_preds[..., :2] *= strides  # to absolute coordinates
        box_preds[..., 2:] = torch.exp(box_preds[..., 2:]) * strides  # wh

        return cls_preds, box_preds


class YOLOv8(nn.Module):
    def __init__(self, num_classes, reg_max=16):
        """
        YOLOv8 implementation

        Args:
            num_classes (int): Number of classes to detect
            reg_max (int): Maximum value for distribution-based regression
        """
        super().__init__()
        self.num_classes = num_classes
        self.backbone = YOLOv8Backbone()
        self.neck = YOLOv8Neck()
        self.head = YOLOv8Head(num_classes, reg_max=reg_max)

    def forward(self, x):
        """
        Forward pass

        Args:
            x: Input tensor of shape (batch_size, 3, H, W)

        Returns:
            List of tuples (cls_predictions, reg_predictions) for each feature level
        """
        # Extract features from backbone
        backbone_features = self.backbone(x)

        # Process through neck
        neck_features = self.neck(backbone_features)

        # Get predictions from head
        predictions = self.head(neck_features)

        if self.training:
            return predictions
        else:
            # Decode predictions for inference
            cls_preds, box_preds = self.head.decode_outputs(predictions)
            return cls_preds, box_preds

    def predict(self, x, conf_threshold=0.25, nms_threshold=0.45):
        """
        Make predictions with non-maximum suppression

        Args:
            x: Input image tensor (batch_size, 3, H, W)
            conf_threshold: Confidence threshold for detections
            nms_threshold: IoU threshold for non-maximum suppression

        Returns:
            List of detected objects with [class_idx, confidence, x1, y1, x2, y2]
        """
        # Set to evaluation mode
        self.eval()

        with torch.no_grad():
            # Forward pass
            cls_preds, box_preds = self.forward(x)

            batch_size = cls_preds.shape[0]
            predictions = []

            # Process each image in batch
            for i in range(batch_size):
                img_cls_preds = cls_preds[i]
                img_box_preds = box_preds[i]

                # Filter by confidence threshold
                max_cls_scores, max_cls_indices = torch.max(img_cls_preds, dim=1)
                mask = max_cls_scores > conf_threshold

                if not mask.any():
                    predictions.append([])  # No detections for this image
                    continue

                # Filter predictions
                filtered_boxes = img_box_preds[mask]
                filtered_scores = max_cls_scores[mask]
                filtered_classes = max_cls_indices[mask]

                # Convert boxes from (cx, cy, w, h) to (x1, y1, x2, y2)
                x1y1 = filtered_boxes[:, :2] - filtered_boxes[:, 2:] / 2
                x2y2 = filtered_boxes[:, :2] + filtered_boxes[:, 2:] / 2
                boxes_xyxy = torch.cat([x1y1, x2y2], dim=1)

                # Apply NMS for each class
                keep_indices = F.nms(boxes_xyxy, filtered_scores, nms_threshold)

                # Get final predictions
                final_boxes = boxes_xyxy[keep_indices]
                final_scores = filtered_scores[keep_indices]
                final_classes = filtered_classes[keep_indices]

                # Prepare results
                img_results = []
                for box, score, cls in zip(final_boxes, final_scores, final_classes):
                    img_results.append([
                        cls.item(),           # Class index
                        score.item(),         # Confidence score
                        box[0].item(),        # x1
                        box[1].item(),        # y1
                        box[2].item(),        # x2
                        box[3].item()         # y2
                    ])

                predictions.append(img_results)

            return predictions

In [ ]:
class DoubleConv(nn.Module):
    """
    Double Convolution block: Conv2d -> BatchNorm -> ReLU -> Conv2d -> BatchNorm -> ReLU
    """
    def __init__(self, in_channels, out_channels, mid_channels=None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels

        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)


class Down(nn.Module):
    """
    Downscaling with maxpool then double conv
    """
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x):
        return self.maxpool_conv(x)


class Up(nn.Module):
    """
    Upscaling then double conv
    """
    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()

        # if bilinear, use the normal convolutions to reduce the number of channels
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1, x2):
        x1 = self.up(x1)

        # Calculate padding required for x1 to match x2 dimensions
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]

        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2,
                        diffY // 2, diffY - diffY // 2])

        # Concatenate along the channels dimension
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)


class OutConv(nn.Module):
    """
    Final 1x1 convolution to map to desired number of output channels
    """
    def __init__(self, in_channels, out_channels):
        super(OutConv, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        return self.conv(x)


class UNet(nn.Module):
    """
    Full U-Net architecture for image segmentation
    """
    def __init__(self, n_channels=3, n_classes=1, bilinear=True, features=[64, 128, 256, 512]):
        super(UNet, self).__init__()
        self.n_channels = n_channels
        self.n_classes = n_classes
        self.bilinear = bilinear

        # Initial double convolution
        self.inc = DoubleConv(n_channels, features[0])

        # Downsampling path
        self.down1 = Down(features[0], features[1])
        self.down2 = Down(features[1], features[2])
        self.down3 = Down(features[2], features[3])

        # Bottleneck - reduce number of channels if using bilinear upsampling
        factor = 2 if bilinear else 1
        self.down4 = Down(features[3], features[3] * 2 // factor)

        # Upsampling path with skip connections
        self.up1 = Up(features[3] * 2, features[3] // factor, bilinear)
        self.up2 = Up(features[3], features[2] // factor, bilinear)
        self.up3 = Up(features[2], features[1] // factor, bilinear)
        self.up4 = Up(features[1], features[0], bilinear)

        # Final 1x1 convolution
        self.outc = OutConv(features[0], n_classes)

    def forward(self, x):
        # Contracting path
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)

        # Expanding path with skip connections
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)

        # Final 1x1 convolution
        logits = self.outc(x)

        return logits


class UNetWithLogits(UNet):
    """
    U-Net with logits output (no activation function)
    Useful for using with BCEWithLogitsLoss or CrossEntropyLoss
    """
    def forward(self, x):
        return super().forward(x)


class UNetWithSigmoid(UNet):
    """
    U-Net with sigmoid activation for binary segmentation
    """
    def forward(self, x):
        return torch.sigmoid(super().forward(x))


class UNetWithSoftmax(UNet):
    """
    U-Net with softmax activation for multi-class segmentation
    """
    def forward(self, x):
        return F.softmax(super().forward(x), dim=1)